# 02 - Preprocesamiento e Ingeniería de Variables

Luego del análisis exploratorio realizado en el Notebook 01, la siguiente etapa consiste en preparar los datos para el entrenamiento de modelos de aprendizaje automático.

El preprocesamiento constituye una fase crítica dentro del flujo de ciencia de datos, ya que garantiza que las variables se encuentren en un formato adecuado para ser utilizadas por las redes neuronales y otros algoritmos predictivos.

En este notebook se implementa un proceso estructurado de preparación de datos, incluyendo tratamiento de valores perdidos, codificación de variables categóricas, escalamiento de variables numéricas y partición del dataset en conjuntos de entrenamiento, validación y prueba.

Asimismo, se desarrolla un proceso de **ingeniería de variables (feature engineering)** con el objetivo de construir nuevas variables derivadas capaces de capturar patrones de comportamiento asociados al abandono de clientes.

El diseño del pipeline busca asegurar **reproducibilidad, reutilización y control de data leakage**, garantizando que el preprocesamiento sea aprendido únicamente a partir del conjunto de entrenamiento y posteriormente aplicado de forma consistente a validación y prueba.

Los objetivos específicos de este notebook son:

- Implementar tratamiento de valores faltantes.
- Codificar variables categóricas mediante OneHotEncoder.
- Escalar variables numéricas utilizando StandardScaler.
- Crear particiones train / validation / test.
- Implementar control de data leakage.
- Construir variables derivadas justificadas mediante feature engineering.
- Guardar el pipeline y datasets procesados para su reutilización en los notebooks posteriores.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

SEED = 42
np.random.seed(SEED)

In [2]:
# cargar dataset

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

df = pd.read_csv(url)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Limpieza inicial
convertir TotalCharges a numérico

In [13]:
df['TotalCharges'] = pd.to_numeric(
    df['TotalCharges'],
    errors='coerce'
)

# revisar missing
df.isnull().sum()

customerID           0
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

In [4]:
#TotalCharges tiene 11 vacíos.
#No los eliminaremos todavía.
#Los trataremos con SimpleImputer, que es más profesional y evita leakage.

In [5]:
#Definir Target y Features
# variable objetivo
y = df['Churn']

# features
X = df.drop(['Churn', 'customerID'], axis=1)

# convertir target
y = y.map({
    'Yes':1,
    'No':0
})

print(X.shape)
print(y.shape)

(7043, 19)
(7043,)


### FEATURE ENGINEERING
5 variables derivadas

In [6]:


X = X.copy()

# 1
X['avg_charge_per_month'] = (
    X['TotalCharges'] /
    (X['tenure'] + 1)
)

# 2
X['is_new_customer'] = (
    X['tenure'] <= 6
).astype(int)

# 3
X['long_term_contract'] = (
    X['Contract'].isin(
        ['One year','Two year']
    )
).astype(int)

# 4
service_cols = [
    'PhoneService',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

for col in service_cols:
    X[col+'_bin'] = X[col].replace({
        'Yes':1,
        'No':0,
        'No internet service':0,
        'No phone service':0
    })

X['service_count'] = X[
    [c+'_bin' for c in service_cols]
].sum(axis=1)

# 5
X['electronic_monthly_risk'] = (
    (X['PaymentMethod']=='Electronic check')
    &
    (X['Contract']=='Month-to-month')
).astype(int)

X.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,long_term_contract,PhoneService_bin,OnlineSecurity_bin,OnlineBackup_bin,DeviceProtection_bin,TechSupport_bin,StreamingTV_bin,StreamingMovies_bin,service_count,electronic_monthly_risk
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,0,0,0,1,0,0,0,0,1,1
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,1,1,1,0,1,0,0,0,3,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,0,1,1,1,0,0,0,0,3,0
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,1,0,1,0,1,1,0,0,3,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,0,1,0,0,0,0,0,0,1,1


### Train / Validation / Test (CONTROL DE LEAKAGE)


In [7]:

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=42,
    stratify=y_train_full
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(5088, 31)
(898, 31)
(1057, 31)


### Preprocessing Pipeline


In [8]:
categorical_cols = X_train.select_dtypes(
    include='string'
).columns

numeric_cols = X_train.select_dtypes(
    include=['int64','float64']
).columns



In [9]:
numeric_transformer = Pipeline([
    (
        'imputer',
        SimpleImputer(strategy='median')
    ),
    (
        'scaler',
        StandardScaler()
    )
])

categorical_transformer = Pipeline([
    (
        'imputer',
        SimpleImputer(
            strategy='most_frequent'
        )
    ),
    (
        'onehot',
        OneHotEncoder(
            handle_unknown='ignore'
        )
    )
])

preprocessor = ColumnTransformer([
    (
        'num',
        numeric_transformer,
        numeric_cols
    ),
    (
        'cat',
        categorical_transformer,
        categorical_cols
    )
])

### Aplicar pipeline


In [10]:
X_train_p = preprocessor.fit_transform(
    X_train
)

X_val_p = preprocessor.transform(
    X_val
)

X_test_p = preprocessor.transform(
    X_test
)

print(X_train_p.shape)
print(X_val_p.shape)
print(X_test_p.shape)

(5088, 49)
(898, 49)
(1057, 49)


In [11]:
# PARA GUARDAR PREPROCESAMIENTO
import os
import joblib

os.makedirs("modelo_final", exist_ok=True)

# guardar pipeline
joblib.dump(
    preprocessor,
    "modelo_final/preprocessor.pkl"
)

# guardar datasets procesados
joblib.dump(
    X_train_p,
    "modelo_final/X_train_p.pkl"
)

joblib.dump(
    X_val_p,
    "modelo_final/X_val_p.pkl"
)

joblib.dump(
    X_test_p,
    "modelo_final/X_test_p.pkl"
)

joblib.dump(
    y_train,
    "modelo_final/y_train.pkl"
)

joblib.dump(
    y_val,
    "modelo_final/y_val.pkl"
)

joblib.dump(
    y_test,
    "modelo_final/y_test.pkl"
)

# guardar columnas originales
joblib.dump(
    X_train.columns.tolist(),
    "modelo_final/input_columns.pkl"
)

print("Pipeline y datasets guardados.")

Pipeline y datasets guardados.
